In [13]:
from supabase import create_client, Client
from dotenv import load_dotenv
import os

load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

# Inicializa el cliente de Supabase
supabase: Client = create_client(url, key)

In [14]:
response = supabase.table('rates_driver').select("*").execute()

import pandas as pd
df = pd.DataFrame(response.data)
print(df.head())

   id  rider_id  driver_id  punctuality  behavior  communication  security  \
0   5         3          1            5         5              5         5   
1   6         4          1            4         5              2         2   
2   7         5          1            1         1              1         1   
3  12         2          3            5         5              5         4   
4  13         3          3            3         5              5         5   

   ride_id  
0        1  
1        1  
2        1  
3        4  
4        4  


In [15]:
result = (
    df
    .assign(
        rating=lambda x: (
            x["punctuality"] + x["communication"] + x["behavior"] + x["security"]
        ) / 4
    )
    .groupby(["rider_id", "driver_id"], as_index=False)
    .agg({
        "rating": "mean"
    })
)

print(result)

     rider_id  driver_id  rating
0           1          2    4.75
1           2          1    5.00
2           2          3    4.75
3           3          1    5.00
4           3          2    4.00
..        ...        ...     ...
995        78         20    2.00
996        78         21    3.00
997        78         23    3.25
998        78         24    2.50
999        78         25    2.00

[1000 rows x 3 columns]


In [16]:
records = result.to_dict(orient="records")

response = (
    supabase
    .table("rider_driver_rate")
    .insert(records)
    .execute()
)

print(response)

data=[{'rider_id': 1, 'driver_id': 2, 'rating': 4.75}, {'rider_id': 2, 'driver_id': 1, 'rating': 5.0}, {'rider_id': 2, 'driver_id': 3, 'rating': 4.75}, {'rider_id': 3, 'driver_id': 1, 'rating': 5.0}, {'rider_id': 3, 'driver_id': 2, 'rating': 4.0}, {'rider_id': 3, 'driver_id': 3, 'rating': 4.5}, {'rider_id': 4, 'driver_id': 1, 'rating': 3.25}, {'rider_id': 4, 'driver_id': 2, 'rating': 3.0}, {'rider_id': 4, 'driver_id': 3, 'rating': 3.75}, {'rider_id': 5, 'driver_id': 1, 'rating': 1.0}, {'rider_id': 5, 'driver_id': 2, 'rating': 4.25}, {'rider_id': 5, 'driver_id': 3, 'rating': 1.75}, {'rider_id': 9, 'driver_id': 8, 'rating': 3.0}, {'rider_id': 9, 'driver_id': 11, 'rating': 2.5}, {'rider_id': 9, 'driver_id': 14, 'rating': 2.0}, {'rider_id': 9, 'driver_id': 15, 'rating': 2.25}, {'rider_id': 9, 'driver_id': 16, 'rating': 2.0}, {'rider_id': 9, 'driver_id': 18, 'rating': 2.5}, {'rider_id': 9, 'driver_id': 19, 'rating': 1.25}, {'rider_id': 9, 'driver_id': 21, 'rating': 2.25}, {'rider_id': 9, 'd